# Qwen3-8B 학습 + 다음 3개 제출본 노트북

## 구성
1. **QLoRA 파인튜닝** — `train.csv` + `train_backtrans.csv` 합산, WandB · Notion 자동 연동
2. **추론 및 제출** — 최고 체크포인트로 `abstract_short` 계열 3종 + MBR 앙상블

생성 파일:
- `submit_11_top1_absShort_result_v3.csv`
- `submit_12_top1_absShort_clean_v3.csv`
- `submit_13_top1_mbr_absShort_best3.csv`


In [1]:
import os, sys
# 학습 시 WandB 활성화 (추론만 실행할 경우 아래 주석 해제)
# os.environ['WANDB_DISABLED'] = 'true'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print('python', sys.version)
try:
    import torch, transformers, peft, bitsandbytes, trl
    print('torch            :', torch.__version__)
    print('cuda available   :', torch.cuda.is_available())
    print('transformers     :', transformers.__version__)
    print('peft             :', peft.__version__)
    print('bitsandbytes     :', bitsandbytes.__version__)
    print('trl              :', trl.__version__)
except Exception as e:
    print('env check error  :', e)


python 3.10.13 (main, Sep 11 2023, 13:44:35) [GCC 11.2.0]
torch            : 2.10.0+cu128
cuda available   : True
transformers     : 5.3.0
peft             : 0.18.1
bitsandbytes     : 0.49.2
trl              : 0.24.0


## 1. QLoRA 파인튜닝 (train.csv + train_backtrans.csv)

| 항목 | 값 |
|---|---|
| 모델 | Qwen/Qwen3-8B |
| LoRA r / alpha | 32 / 64 |
| 배치 | 2 × grad_accum 4 = 실효 8 |
| 학습률 | 1e-4 (cosine) |
| 에폭 | 3 |
| 데이터 | train.csv + train_backtrans.csv (없으면 원본만) |
| 로깅 | WandB (`dialogue-summarization`) + Notion |


In [8]:
# -*- coding: utf-8 -*-
# ================================================================
# QLoRA 파인튜닝 — Qwen3-8B | 합산 데이터 | WandB · Notion 연동
# ================================================================
import gc, os, re, sys, json, time, traceback
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# .env 로드 (WANDB_API_KEY, NOTION_TOKEN, NOTION_DATABASE_ID)
try:
    from dotenv import load_dotenv
    for _env in [
        '/root/upstage-nlp-nlp/code/.env',
        '/data/ephemeral/home/project/nlp-team2/dialogue-summarization/.env',
    ]:
        if Path(_env).exists():
            load_dotenv(_env)
            print(f'.env 로드: {_env}')
            break
except Exception as _e:
    print(f'dotenv 로드 실패: {_e}')

# ================================================================
# CONFIG  ← 필요시 이 블록만 수정
# ================================================================
DATA_DIR             = Path('/data/ephemeral/home/project/nlp-team2/dialogue-summarization/data/raw')
TRAIN_PATH           = DATA_DIR / 'train.csv'
TRAIN_BACKTRANS_PATH = DATA_DIR / 'train_backtrans.csv'  # 없으면 원본만 사용
DEV_PATH             = DATA_DIR / 'dev.csv'

MODEL_NAME     = 'Qwen/Qwen3-8B'
LORA_R         = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.05
MAX_SEQ_LEN    = 1536
NUM_EPOCHS     = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 4       # 실효 배치 = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE  = 1e-4
WARMUP_RATIO   = 0.05
WEIGHT_DECAY   = 0.01

WANDB_PROJECT  = 'dialogue-summarization'
WANDB_ENTITY   = 'sskim7415-'          # solar_config.yaml 과 동일
TAGS           = ['Qwen3-8B', 'QLoRA', 'backtrans', 'causal-lm']

CKPT_ROOT  = Path('/root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TS     = datetime.now().strftime('%Y%m%d-%H%M%S')
RUN_NAME   = f'qwen3-8b-backtrans-r{LORA_R}-lr{LEARNING_RATE}-ep{NUM_EPOCHS}-{RUN_TS}'
OUTPUT_DIR = CKPT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    '당신은 한국어 대화 요약 전문가입니다. '
    '대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다. '
    '요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요. '
    '핵심 내용만 1~3문장으로 간결하게 요약하세요. '
    '반드시 한국어로 작성하되, CPU·USB 등 고유 영문 약어나 전문 용어는 그대로 사용해도 됩니다.'
)

# ================================================================
# 유틸
# ================================================================
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def pick_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=pick_dtype(),
    )

def normalize_text(text):
    text = str(text)
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<\|.*?\|>', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'#\s*Person\s*(\d+)\s*#', r'#Person\1#', text)
    text = re.sub(r'^요약\s*:\s*', '', text).strip()
    return re.sub(r'\s+', ' ', text).strip() or '빈 요약'

def format_sample(row, tokenizer):
    """dialogue + summary → chat template 적용 문자열 반환"""
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': (
            '아래 대화를 읽고 핵심 내용을 한국어로 요약해주세요. '
            '화자 태그(#Person1# 등)를 유지하세요.\n\n' + str(row['dialogue'])
        )},
        {'role': 'assistant', 'content': str(row.get('summary', ''))},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
    )

# ================================================================
# 1. 데이터 로드 및 합산
# ================================================================
print('=== 데이터 로드 ===')
train_df = pd.read_csv(TRAIN_PATH)
print(f'  원본 train       : {len(train_df):,}행')

if TRAIN_BACKTRANS_PATH.exists():
    bt_df = pd.read_csv(TRAIN_BACKTRANS_PATH)
    # 백번역본에 summary 컬럼이 없으면 원본 summary 재사용
    if 'summary' not in bt_df.columns and 'summary' in train_df.columns:
        bt_df = bt_df.copy()
        bt_df['summary'] = train_df['summary'].values[:len(bt_df)]
    train_combined = pd.concat([train_df, bt_df], ignore_index=True)
    print(f'  백번역본         : {len(bt_df):,}행')
    print(f'  합산 total       : {len(train_combined):,}행')
else:
    train_combined = train_df
    print(f'  ※ train_backtrans.csv 없음 → 원본만 사용 ({len(train_combined):,}행)')

dev_df = pd.read_csv(DEV_PATH)
print(f'  dev              : {len(dev_df):,}행')

# ================================================================
# 2. 토크나이저 + 데이터셋 변환
# ================================================================
print('\n=== 토크나이저 로드 ===')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

train_texts   = [format_sample(row, tokenizer) for _, row in train_combined.iterrows()]
train_dataset = Dataset.from_dict({'text': train_texts})
print(f'  학습 샘플: {len(train_dataset):,}')

# ================================================================
# 3. WandB 초기화
# ================================================================
WANDB_ENABLED = False
try:
    import wandb
    _api_key = os.environ.get('WANDB_API_KEY')
    if _api_key:
        wandb.login(key=_api_key)
    else:
        wandb.login()  # 이미 로그인된 세션 사용
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        tags=TAGS,
        config={
            'model': MODEL_NAME,
            'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA, 'lora_dropout': LORA_DROPOUT,
            'max_seq_length': MAX_SEQ_LEN,
            'num_epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE,
            'grad_accum': GRAD_ACCUM, 'learning_rate': LEARNING_RATE,
            'train_samples': len(train_dataset),
            'use_backtrans': TRAIN_BACKTRANS_PATH.exists(),
        },
        resume='allow',
    )
    WANDB_ENABLED = True
    print(f'WandB 연결: {wandb_run.url}')
except Exception as _e:
    print(f'WandB 비활성화: {_e}')

# ================================================================
# 4. Notion 실험 기록 시작
# ================================================================
NOTION_ENABLED  = False
notion_page_id  = None
try:
    from notion_client import Client as NotionClient
    notion = NotionClient(auth=os.environ['NOTION_TOKEN'])
    _db_id = os.environ['NOTION_DATABASE_ID']
    _resp  = notion.pages.create(
        parent={'database_id': _db_id},
        properties={
            '실험명':   {'title': [{'text': {'content': RUN_NAME}}]},
            '상태':     {'status': {'name': '진행 중'}},
            '실험일':   {'date': {'start': datetime.now(timezone.utc).date().isoformat()}},
            'Run Name': {'rich_text': [{'text': {'content': RUN_NAME}}]},
            'Dataset':  {'rich_text': [{'text': {'content': 'DialogSum-KO + BackTrans'}}]},
            'Task':     {'select': {'name': 'Summarization'}},
            '목적':     {'rich_text': [{'text': {'content': 'Qwen3-8B QLoRA 파인튜닝 (backtrans 데이터 증강)'}}]},
            '태그':     {'multi_select': [{'name': t} for t in TAGS]},
        },
    )
    notion_page_id = _resp['id']
    NOTION_ENABLED = True
    print(f'Notion 기록 시작 (page_id={notion_page_id})')
except Exception as _e:
    print(f'Notion 비활성화: {_e}')

# ================================================================
# 5. 모델 로드 + LoRA 적용
# ================================================================
print('\n=== 모델 로드 ===')
cleanup()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=get_bnb_config(),
    device_map={'': 0},
    torch_dtype=pick_dtype(),        # ← dtype= 아님
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# ================================================================
# 6. SFTTrainer 학습
# ================================================================
print('\n=== 학습 시작 ===')
_train_start = time.time()

sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    report_to='wandb' if WANDB_ENABLED else 'none',
    run_name=RUN_NAME if WANDB_ENABLED else None,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    max_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,
    seed=42,
)

try:
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=sft_cfg,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / 'final'))
    tokenizer.save_pretrained(str(OUTPUT_DIR / 'final'))

    _elapsed = time.time() - _train_start
    print(f'\n학습 완료: {_elapsed / 60:.1f}분')
    print(f'저장 경로: {OUTPUT_DIR / "final"}')

    # ================================================================
    # 7. Dev ROUGE 평가
    # ================================================================
    from rouge import Rouge as RougeScorer
    _rouge = RougeScorer()

    print('\n=== Dev ROUGE 평가 ===')
    model.eval()
    if torch.cuda.is_bf16_supported():
        model = model.to(torch.bfloat16)

    _DEV_SYS = (
        '당신은 한국어 대화 요약 전문가입니다. '
        '대화의 핵심 사건만 아주 짧게 요약하세요. '
        '반드시 #Person1#, #Person2# 같은 화자 태그를 유지하세요. '
        '1문장만 출력하고, 불필요한 수식어는 쓰지 마세요.'
    )
    _preds, _refs = [], []
    for _, _row in dev_df.iterrows():
        _msgs = [
            {'role': 'system', 'content': _DEV_SYS},
            {'role': 'user',   'content': (
                '다음 대화를 1문장으로만 매우 간결하게 요약하세요. '
                '핵심 행동/결정만 남기고 화자 태그는 유지하세요.\n\n' + str(_row['dialogue'])
            )},
        ]
        _prompt = tokenizer.apply_chat_template(
            _msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        _inputs = tokenizer(
            _prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN
        )
        _inputs = {k: v.to(model.device) for k, v in _inputs.items()}
        with torch.no_grad():
            _out = model.generate(
                **_inputs, max_new_tokens=90, do_sample=False,
                repetition_penalty=1.05, use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        _pred = tokenizer.decode(_out[0][_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        _preds.append(normalize_text(_pred))
        _refs.append(normalize_text(_row['summary']))

    _sc    = _rouge.get_scores(_preds, _refs, avg=True)
    rouge1 = _sc['rouge-1']['f']
    rouge2 = _sc['rouge-2']['f']
    rougeL = _sc['rouge-l']['f']
    final  = rouge1 + rouge2 + rougeL
    print(f'  rouge-1 : {rouge1:.4f}')
    print(f'  rouge-2 : {rouge2:.4f}')
    print(f'  rouge-L : {rougeL:.4f}')
    print(f'  합계    : {final:.4f}')

    # ================================================================
    # 8. checkpoint_eval_summary.csv 업데이트
    # ================================================================
    _summary_csv = CKPT_ROOT / 'checkpoint_eval_summary.csv'
    _new_row = pd.DataFrame([{
        'checkpoint': str(OUTPUT_DIR / 'final'),
        'rouge1': rouge1, 'rouge2': rouge2, 'rougeL': rougeL,
        'final': final, 'run_name': RUN_NAME,
    }])
    if _summary_csv.exists():
        _prev = pd.read_csv(_summary_csv)
        _merged = pd.concat([_new_row, _prev], ignore_index=True).sort_values('final', ascending=False)
    else:
        _merged = _new_row
    _merged.to_csv(_summary_csv, index=False)
    print(f'checkpoint_eval_summary.csv 업데이트 완료 (1위: {_merged.iloc[0]["run_name"]})')

    # ================================================================
    # 9. WandB / Notion 결과 기록
    # ================================================================
    _result_summary = (
        f'rouge1={rouge1:.4f}, rouge2={rouge2:.4f}, rougeL={rougeL:.4f}, 합계={final:.4f} | '
        f'학습시간={_elapsed/60:.1f}분 | '
        f'데이터={len(train_dataset):,}샘플 (backtrans={TRAIN_BACKTRANS_PATH.exists()})'
    )
    if WANDB_ENABLED:
        import wandb
        wandb.log({'dev/rouge1': rouge1, 'dev/rouge2': rouge2,
                   'dev/rougeL': rougeL, 'dev/final': final})
        wandb.run.summary.update({'dev_rouge1': rouge1, 'dev_rouge2': rouge2,
                                   'dev_rougeL': rougeL, 'dev_final': final})
        wandb.finish()
        print('WandB 기록 완료')

    if NOTION_ENABLED and notion_page_id:
        notion.pages.update(
            page_id=notion_page_id,
            properties={
                '상태':         {'status': {'name': '완료'}},
                'rouge1':       {'number': round(rouge1, 4)},
                'rouge2':       {'number': round(rouge2, 4)},
                'rougeL':       {'number': round(rougeL, 4)},
                'final_result': {'number': round(final, 4)},
                '결과 요약':    {'rich_text': [{'text': {'content': _result_summary[:2000]}}]},
            },
        )
        print('Notion 기록 완료')

except Exception as _e:
    _tb = traceback.format_exc()
    print(f'[ERROR] 학습 실패:\n{_tb}')
    if WANDB_ENABLED:
        import wandb
        wandb.finish(exit_code=1)
    if NOTION_ENABLED and notion_page_id:
        notion.pages.update(
            page_id=notion_page_id,
            properties={
                '상태':      {'status': {'name': '실패'}},
                '결과 요약': {'rich_text': [{'text': {'content': _tb[:2000]}}]},
            },
        )
    raise
finally:
    cleanup()


.env 로드: /data/ephemeral/home/project/nlp-team2/dialogue-summarization/.env
=== 데이터 로드 ===
  원본 train       : 12,457행
  백번역본         : 12,457행
  합산 total       : 24,914행
  dev              : 499행

=== 토크나이저 로드 ===


wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id y5xcqpu2.


  학습 샘플: 24,914


Problem at: /tmp/ipykernel_905379/3438661494.py 163 <module>
WandB 비활성화: Run initialization has timed out after 90.0 sec. 
Please refer to the documentation for additional information: https://docs.wandb.ai/guides/track/tracking-faq#initstarterror-error-communicating-with-wandb-process-
Notion 기록 시작 (page_id=3355930a-a61a-816b-860c-e20cfc52a28f)

=== 모델 로드 ===


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

trainable params: 87,293,952 || all params: 8,278,029,312 || trainable%: 1.0545

=== 학습 시작 ===


TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'

## 2. 추론 및 제출 파일 생성

`checkpoint_eval_summary.csv`에서 1위 체크포인트를 자동 선택하여 3종 추론 후 MBR 앙상블 제출본을 생성합니다.


In [3]:
# -*- coding: utf-8 -*-
import gc
from pathlib import Path
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from rouge import Rouge

# ============================================================
# 호환성 패치
# ============================================================
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        atoms = target.split(".")
        parent = self
        for item in atoms[:-1]:
            parent = parent.get_submodule(item)
        setattr(parent, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

# ============================================================
# CONFIG
# ============================================================
DATA_DIR = Path("/data/ephemeral/home/project/nlp-team2/dialogue-summarization/data/raw")
PRED_ROOT = Path("/root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b")
MANUAL_DIR = PRED_ROOT / "manual_submissions"
CACHE_DIR = MANUAL_DIR / "cache_best3_round"
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_SEQ_LENGTH = 1536
DEFAULT_MAX_NEW_TOKENS = 90
TIGHT_MAX_NEW_TOKENS = 72

PROMPTS = {
    "abstract_short_base": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "대화의 핵심 사건만 아주 짧게 요약하세요. "
            "반드시 #Person1#, #Person2# 같은 화자 태그를 유지하세요. "
            "1문장만 출력하고, 불필요한 수식어는 쓰지 마세요."
        ),
        "user": (
            "다음 대화를 1문장으로만 매우 간결하게 요약하세요. "
            "핵심 행동/결정만 남기고 화자 태그는 유지하세요.\n\n{dialogue}"
        ),
        "max_new_tokens": DEFAULT_MAX_NEW_TOKENS,
    },
    "abstract_short_result_v3": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "대화에서 실제로 일어난 핵심 행동, 요청, 제안, 합의, 결정만 남겨 1문장으로 요약하세요. "
            "반드시 #Person1#, #Person2# 화자 태그를 유지하고, 배경 설명, 감상, 부연 설명은 제거하세요. "
            "짧고 단정한 완전한 문장 1개만 출력하세요."
        ),
        "user": (
            "다음 대화를 읽고 결과 중심으로 1문장 요약하세요. "
            "누가 무엇을 요청/제안/결정/수행했는지만 남기고 나머지는 제거하세요.\n\n{dialogue}"
        ),
        "max_new_tokens": DEFAULT_MAX_NEW_TOKENS,
    },
    "abstract_short_clean_v3": {
        "system": (
            "당신은 한국어 대화 요약 전문가입니다. "
            "대화 전체를 대표하는 핵심 사실만 1문장으로 매우 간결하게 요약하세요. "
            "#Person1#, #Person2# 화자 태그는 반드시 유지하고, 중복 표현, 세부 묘사, 배경 설명은 제거하세요. "
            "문장은 짧고 자연스럽게 끝내세요."
        ),
        "user": (
            "다음 대화를 1문장으로 깔끔하게 요약하세요. "
            "핵심 주제와 결과만 남기고, 군더더기 없이 짧게 작성하세요.\n\n{dialogue}"
        ),
        "max_new_tokens": TIGHT_MAX_NEW_TOKENS,
    },
}

ROUGE = Rouge()

# ============================================================
# 유틸
# ============================================================
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def pick_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def get_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=pick_dtype(),
    )


def normalize_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "빈 요약"


def pairwise_rouge1_agreement(candidates):
    scores = []
    for j, cj in enumerate(candidates):
        ssum, cnt = 0.0, 0
        for k, ck in enumerate(candidates):
            if j == k:
                continue
            try:
                s = ROUGE.get_scores([normalize_text(cj)], [normalize_text(ck)])[0]
                ssum += s["rouge-1"]["f"]
                cnt += 1
            except Exception:
                pass
        scores.append(ssum / max(cnt, 1))
    return scores


def mbr_ensemble(pred_lists):
    names = list(pred_lists.keys())
    n = len(next(iter(pred_lists.values())))
    outputs = []
    selected = {name: 0 for name in names}
    for i in range(n):
        candidates = [pred_lists[name][i] for name in names]
        agreements = pairwise_rouge1_agreement(candidates)
        best_idx = int(np.argmax(agreements))
        name = names[best_idx]
        selected[name] += 1
        outputs.append(candidates[best_idx])
    return outputs, selected


def save_submission(df, preds, out_csv: Path):
    sub = pd.DataFrame({"fname": df["fname"], "summary": preds})
    sub.to_csv(out_csv, index=False)
    print("saved:", out_csv)


def get_top1_checkpoint():
    summary_csv = PRED_ROOT / "checkpoint_eval_summary.csv"
    if not summary_csv.exists():
        raise FileNotFoundError(f"missing: {summary_csv}")
    rank_df = pd.read_csv(summary_csv)
    top1 = Path(rank_df.iloc[0]["checkpoint"])
    print("top1 checkpoint:", top1)
    return top1

# ============================================================
# 모델 로딩 / 생성
# ============================================================
def load_model_for_inference(adapter_path: Path):
    cleanup()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=get_bnb_config(),
        device_map={"": 0},
        torch_dtype=pick_dtype(),   # dtype= 아님
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=False)
    model.eval()
    model.config.use_cache = True
    return model, tokenizer


def generate_summary(model, tokenizer, dialogue: str, prompt_name: str) -> str:
    cfg = PROMPTS[prompt_name]
    messages = [
        {"role": "system", "content": cfg["system"]},
        {"role": "user", "content": cfg["user"].format(dialogue=dialogue)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    prompt = "/no_think\n" + prompt

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.get("max_new_tokens", DEFAULT_MAX_NEW_TOKENS),
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True)
    return normalize_text(summary)


def run_predictions(model, tokenizer, df, prompt_name, out_csv: Path):
    if out_csv.exists():
        cached = pd.read_csv(out_csv)
        if len(cached) == len(df):
            print("cache hit:", out_csv)
            return [normalize_text(x) for x in cached["summary"].tolist()]

    preds = []
    for i, row in enumerate(df.itertuples(index=False), start=1):
        preds.append(generate_summary(model, tokenizer, row.dialogue, prompt_name))
        if i % 50 == 0:
            print(f"  - {prompt_name}: {i}/{len(df)} done")

    save_submission(df, preds, out_csv)
    return preds


def load_existing_submission(csv_path: Path, expected_len: int):
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if len(df) == expected_len and "summary" in df.columns:
            print("reuse existing submission:", csv_path)
            return [normalize_text(x) for x in df["summary"].tolist()]
    return None

# ============================================================
# 실행
# ============================================================
MANUAL_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

test_df = pd.read_csv(DATA_DIR / "test.csv")
top1_ckpt = get_top1_checkpoint()
model, tokenizer = load_model_for_inference(top1_ckpt)

# 1) result-focused abstract_short
pred_result = run_predictions(
    model, tokenizer, test_df, "abstract_short_result_v3",
    MANUAL_DIR / "submit_11_top1_absShort_result_v3.csv"
)

# 2) cleaner/tighter abstract_short
pred_clean = run_predictions(
    model, tokenizer, test_df, "abstract_short_clean_v3",
    MANUAL_DIR / "submit_12_top1_absShort_clean_v3.csv"
)

# 3) strongest expected MBR = previous best abstract_short + result_v3 + clean_v3
prev_best = load_existing_submission(
    MANUAL_DIR / "submit_05_top1_abstract_short.csv",
    len(test_df),
)
if prev_best is None:
    prev_best = run_predictions(
        model, tokenizer, test_df, "abstract_short_base",
        CACHE_DIR / "cache_prev_best_abstract_short.csv"
    )

pred_mbr, selected = mbr_ensemble({
    "prev_best_abstract_short": prev_best,
    "result_v3": pred_result,
    "clean_v3": pred_clean,
})
save_submission(test_df, pred_mbr, MANUAL_DIR / "submit_13_top1_mbr_absShort_best3.csv")
print("MBR selected count:", selected)

try:
    del model, tokenizer
except Exception:
    pass
cleanup()

print("\n===== best-3 next submissions ready =====")
for p in sorted(MANUAL_DIR.glob("submit_1*.csv")):
    print(p)


top1 checkpoint: /root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/qwen3-8b-backtrans-r32-lr0.0001-ep3-20260401-081433/final


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

  - abstract_short_result_v3: 50/499 done
  - abstract_short_result_v3: 100/499 done
  - abstract_short_result_v3: 150/499 done
  - abstract_short_result_v3: 200/499 done
  - abstract_short_result_v3: 250/499 done
  - abstract_short_result_v3: 300/499 done
  - abstract_short_result_v3: 350/499 done
  - abstract_short_result_v3: 400/499 done
  - abstract_short_result_v3: 450/499 done
saved: /root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/manual_submissions/submit_11_top1_absShort_result_v3.csv
  - abstract_short_clean_v3: 50/499 done
  - abstract_short_clean_v3: 100/499 done
  - abstract_short_clean_v3: 150/499 done
  - abstract_short_clean_v3: 200/499 done
  - abstract_short_clean_v3: 250/499 done
  - abstract_short_clean_v3: 300/499 done
  - abstract_short_clean_v3: 350/499 done
  - abstract_short_clean_v3: 400/499 done
  - abstract_short_clean_v3: 450/499 done
saved: /root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/manual_submissio